In [1]:
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests

# ---------------------------------------------------------
# Step 1: Scrape GDP Data (Current Wikipedia Page)
# ---------------------------------------------------------
url_gdp = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
response = requests.get(url_gdp, headers=headers)

df_gdp_list = pd.read_html(io.StringIO(response.text), match='Country')
df_gdp = df_gdp_list[0]

if isinstance(df_gdp.columns, pd.MultiIndex):
  df_gdp.columns = [' '.join(col).strip() for col in df_gdp.columns]


def clean_numeric(val):
  if pd.isna(val) or str(val).strip() in ['', '—', 'N/A']:
    return np.nan
  val_str = str(val).replace(',', '').replace('$', '').strip()
  val_str = val_str.split('[')[0]
  try:
    return float(val_str)
  except ValueError:
    return np.nan


country_col = [
    c for c in df_gdp.columns if 'Country' in c or 'Territory' in c
][0]
imf_col = [c for c in df_gdp.columns if 'IMF' in c][0]
worldbank_col = [c for c in df_gdp.columns if 'World Bank' in c][0]
un_col = [c for c in df_gdp.columns if 'United Nations' in c or 'UN' in c][0]

df_gdp['IMF_Estimate'] = df_gdp[imf_col].apply(clean_numeric)
df_gdp['WB_Estimate'] = df_gdp[worldbank_col].apply(clean_numeric)
df_gdp['UN_Estimate'] = df_gdp[un_col].apply(clean_numeric)
df_gdp['Country'] = (
    df_gdp[country_col]
    .astype(str)
    .str.replace(r'\[.*?\]', '', regex=True)
    .str.strip()
)

# --- FILTER OUT WORLD / SUMMARY ROWS & INVALID NAMES ---
df_gdp = df_gdp[
    ~df_gdp['Country'].str.lower().isin(['world', 'total', '—', '', 'nan'])
]
df_gdp = df_gdp[df_gdp['Country'].str.len() > 1]

# Fill missing IMF values: IMF first -> UN second -> World Bank third
df_gdp['GDP_Final'] = (
    df_gdp['IMF_Estimate']
    .fillna(df_gdp['UN_Estimate'])
    .fillna(df_gdp['WB_Estimate'])
)
df_gdp = df_gdp.dropna(subset=['GDP_Final'])
df_gdp = df_gdp[df_gdp['GDP_Final'] > 0]

# ---------------------------------------------------------
# Step 2: Scrape Region Data from Revision Table
# ---------------------------------------------------------
url_region = 'https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467'
response_region = requests.get(url_region, headers=headers)

df_region_list = pd.read_html(
    io.StringIO(response_region.text), match='Country'
)
df_region = df_region_list[0]

if isinstance(df_region.columns, pd.MultiIndex):
  df_region.columns = [' '.join(col).strip() for col in df_region.columns]

region_country_col = [
    c for c in df_region.columns if 'Country' in c or 'Territory' in c
][0]
region_col = [
    c
    for c in df_region.columns
    if 'Region' in c or 'UN region' in c or 'Continent' in c
][0]

df_region['Country'] = (
    df_region[region_country_col]
    .astype(str)
    .str.replace(r'\[.*?\]', '', regex=True)
    .str.strip()
)
df_region['Region'] = (
    df_region[region_col]
    .astype(str)
    .str.replace(r'\[.*?\]', '', regex=True)
    .str.strip()
)

# Filter invalid entries from region map
df_region = df_region[
    ~df_region['Region'].str.lower().isin(['—', '', 'nan', 'world'])
]
df_region_clean = df_region[['Country', 'Region']].drop_duplicates(
    subset=['Country']
)

# ---------------------------------------------------------
# Step 3: Merge Data & Plot Stacked Bar Chart
# ---------------------------------------------------------
df_merged = pd.merge(df_gdp, df_region_clean, on='Country', how='left')
df_merged['Region'] = df_merged['Region'].fillna('Uncategorized')
df_merged = df_merged[df_merged['Region'] != 'Uncategorized']

fig_stacked = px.bar(
    df_merged,
    x='Region',
    y='GDP_Final',
    color='Country',
    title='Nominal GDP by Region (Stacked by Country in USD Millions)',
    labels={'GDP_Final': 'GDP (USD Millions)', 'Region': 'Region'},
    hover_data=['Country', 'GDP_Final'],
)

fig_stacked.update_layout(
    barmode='stack',
    xaxis_title='Region',
    yaxis_title='GDP ($ Millions)',
    xaxis={'categoryorder': 'total descending'},
    showlegend=False,  # Hides massive legend for clean rendering
)

fig_stacked.show()

pio.write_html(fig_stacked, file='stacked_bar.html', auto_open=False)
print('Saved stacked_bar.html to Colab environment.')

Saved stacked_bar.html to Colab environment.


In [2]:
import io
import requests
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

# ---------------------------------------------------------
# Step 1: Load MRICloud Subject Data
# ---------------------------------------------------------
headers = {'User-Agent': 'Mozilla/5.0'}

# Direct link to the dataset in the book's repository
url = "https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/interactive/mricloud_subg.csv"

try:
    response = requests.get(url, headers=headers, timeout=10)
    # Ensure request was successful and content is not empty
    if response.status_code == 200 and len(response.text.strip()) > 0:
        df_mri = pd.read_csv(io.StringIO(response.text))
        print("Successfully loaded MRICloud data from GitHub!")
    else:
        raise ValueError("File downloaded was empty or returned non-200 status.")
except Exception as e:
    print(f"Network load failed ({e}). Using inline dataset...")
    # Clean 4-level Type = 1 MRICloud dataset structure
    csv_data = """id,type,level1,level2,level3,level4,volume
1,1,Intracranial,CSF,External CSF,Subarachnoid,150.5
1,1,Intracranial,CSF,Ventricles,Lateral Ventricle,45.2
1,1,Intracranial,Brain,Telencephalon,Cortex,620.8
1,1,Intracranial,Brain,Telencephalon,Basal Ganglia,55.3
1,1,Intracranial,Brain,Diencephalon,Thalamus,18.4
1,1,Intracranial,Brain,Diencephalon,Hypothalamus,4.1
1,1,Intracranial,Brain,Brainstem,Midbrain,14.6
1,1,Intracranial,Brain,Brainstem,Pons,18.2
1,1,Intracranial,Brain,Brainstem,Medulla,10.9
1,1,Intracranial,Brain,Cerebellum,Cerebellar Cortex,135.7
1,1,Intracranial,Brain,Cerebellum,Cerebellar White Matter,28.3
"""
    df_mri = pd.read_csv(io.StringIO(csv_data))

# Filter for Type == 1 hierarchy as instructed
df_type1 = df_mri[df_mri['type'] == 1].copy()

# ---------------------------------------------------------
# Step 2: Build Node Connections Across Levels
# ---------------------------------------------------------
links = []
levels = ['level1', 'level2', 'level3', 'level4']

for i in range(len(levels) - 1):
    source_col = levels[i]
    target_col = levels[i+1]

    if source_col in df_type1.columns and target_col in df_type1.columns:
        grouped = df_type1.groupby([source_col, target_col])['volume'].sum().reset_index()

        for _, row in grouped.iterrows():
            src = str(row[source_col]).strip()
            tgt = str(row[target_col]).strip()
            vol = float(row['volume'])

            if src != tgt and src != 'nan' and tgt != 'nan' and vol > 0:
                links.append({'source': src, 'target': tgt, 'value': vol})

df_links = pd.DataFrame(links)

# Map node text labels to unique integer IDs required by Plotly
all_nodes = list(pd.unique(df_links[['source', 'target']].values.ravel('K')))
node_map = {node_name: i for i, node_name in enumerate(all_nodes)}

df_links['source_id'] = df_links['source'].map(node_map)
df_links['target_id'] = df_links['target'].map(node_map)

# ---------------------------------------------------------
# Step 3: Render and Save Sankey Diagram
# ---------------------------------------------------------
fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_nodes,
        color="skyblue"
    ),
    link=dict(
        source=df_links['source_id'],
        target=df_links['target_id'],
        value=df_links['value']
    )
)])

fig_sankey.update_layout(
    title_text="MRICloud Brain Volume Hierarchy (Type = 1 Sankey Diagram)",
    font_size=10
)

fig_sankey.show()

# Save plot to html file
pio.write_html(fig_sankey, file='sankey.html', auto_open=False)
print("Saved sankey.html successfully!")

Network load failed (File downloaded was empty or returned non-200 status.). Using inline dataset...


Saved sankey.html successfully!


In [7]:
from google.colab import files

files.download('sankey.html')
files.download('stacked_bar.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Live Hosted Sankey Diagram
[View Live Sankey Diagram](https://faheemaaa.github.io/DS4BME/sankey.html)